# dARK Core Minter API - Unified Integration Notebook

This notebook provides a single, readable integration path for the **current** Minter API behavior.

It intentionally focuses on the main flows that matter in day-to-day validation:
- service smoke checks
- authority bootstrap and authorization
- reserve and fetch
- update to `DRAFT` with JSON and XML metadata
- representative validation checks
- batch reserve
- tombstone lifecycle
- optional metadata-worker and chain-worker publish/update
- optional chain import from on-chain state

The focused notebooks remain useful for deeper or stress-oriented exploration, but this one is intended to be the easiest place to start.


## Runtime Variables

The notebook reads configuration from environment variables, following the same pattern as the other service notebooks.

- `MINTER_BASE_URL` default: `http://localhost:8001`
- `ADMIN_API_URL` default: `http://localhost:8000/api/v1/admin`
- `AUTHORITY_ID` default: autogenerated
- `NAAN` default: `12345`
- `UNAUTHORIZED_NAAN` default: `99999`
- `REGISTER_AUTHORITY` default: `true`
- `EXISTING_CHAIN_ARK` default: empty
- `POLL_INTERVAL_SECONDS` default: `2`
- `POLL_TIMEOUT_SECONDS` default: `90`

**Worker update (2026-05):**
- `/api/v1/worker/status` uses `alive` for runtime health and `state` for `idle`/`backlogged`/`stale`.
- `last_cycle.processed` means ARKs attempted in the last cycle, not total processed.
- Metadata and chain workers are separate; chain publication uses the fast path and only reconciles on errors.


In [ ]:
import json
import os
import time
import uuid
from datetime import datetime, timezone
from pprint import pprint

import requests

# -----------------------------------------------------------------------------
# Runtime configuration
# -----------------------------------------------------------------------------
MINTER_BASE_URL = os.getenv("MINTER_BASE_URL", "http://localhost:8001").rstrip("/")
MINTER_API_V1 = f"{MINTER_BASE_URL}/api/v1"
ADMIN_API_URL = os.getenv("ADMIN_API_URL", "http://localhost:8000/api/v1/admin").rstrip("/")

AUTHORITY_ID = os.getenv("AUTHORITY_ID", f"test-authority-{int(time.time())}")
NAAN = os.getenv("NAAN", "12345").strip()
UNAUTHORIZED_NAAN = os.getenv("UNAUTHORIZED_NAAN", "99999").strip()

REGISTER_AUTHORITY = os.getenv("REGISTER_AUTHORITY", "true").lower() in {"1", "true", "yes", "y"}
EXISTING_CHAIN_ARK = os.getenv("EXISTING_CHAIN_ARK", "").strip() or None

POLL_INTERVAL_SECONDS = float(os.getenv("POLL_INTERVAL_SECONDS", "2"))
POLL_TIMEOUT_SECONDS = int(os.getenv("POLL_TIMEOUT_SECONDS", "90"))

print("MINTER_BASE_URL:", MINTER_BASE_URL)
print("MINTER_API_V1:", MINTER_API_V1)
print("ADMIN_API_URL:", ADMIN_API_URL)
print("AUTHORITY_ID:", AUTHORITY_ID)
print("NAAN:", NAAN)
print("UNAUTHORIZED_NAAN:", UNAUTHORIZED_NAAN)
print("REGISTER_AUTHORITY:", REGISTER_AUTHORITY)
print("EXISTING_CHAIN_ARK:", EXISTING_CHAIN_ARK)
print("POLL_INTERVAL_SECONDS:", POLL_INTERVAL_SECONDS)
print("POLL_TIMEOUT_SECONDS:", POLL_TIMEOUT_SECONDS)

# -----------------------------------------------------------------------------
# Test bookkeeping
# -----------------------------------------------------------------------------
RESULTS = []
CONTEXT = {}


def record(name: str, status: str, detail: str = ""):
    status = status.upper()
    RESULTS.append({"name": name, "status": status, "detail": detail})
    print(f"[{status}] {name}")
    if detail:
        print("       ", detail)


def check(name: str, condition: bool, ok_detail: str = "", fail_detail: str = ""):
    record(name, "PASS" if condition else "FAIL", ok_detail if condition else fail_detail)


def skip(name: str, detail: str = ""):
    record(name, "SKIP", detail)


def summarize_results():
    passed = sum(1 for row in RESULTS if row["status"] == "PASS")
    failed = sum(1 for row in RESULTS if row["status"] == "FAIL")
    skipped = sum(1 for row in RESULTS if row["status"] == "SKIP")
    total = len(RESULTS)

    print("\n=== TEST SUMMARY ===")
    print(f"Total checks : {total}")
    print(f"Passed       : {passed}")
    print(f"Failed       : {failed}")
    print(f"Skipped      : {skipped}")

    if failed:
        print("\nFailed checks:")
        for row in RESULTS:
            if row["status"] == "FAIL":
                print(f"- {row['name']}: {row['detail']}")

    return {"total": total, "passed": passed, "failed": failed, "skipped": skipped}


def decode_body(resp: requests.Response):
    ctype = (resp.headers.get("content-type") or "").lower()
    if "application/json" in ctype:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        return resp.json()
    except Exception:
        return resp.text


MUTATING_METHODS = {"POST", "PUT", "PATCH", "DELETE"}
DEFAULT_AUTHORITY_HEADER = "X-Authority-Id"


def _with_authority_headers(method: str, base: str, kwargs: dict) -> dict:
    """Inject authority identity headers for mutating minter API calls in local/dev mode."""
    method = method.upper()
    if method not in MUTATING_METHODS or base != MINTER_API_V1:
        return kwargs

    merged = dict(kwargs)
    headers = dict(merged.get("headers") or {})
    if any(headers.get(name) for name in ("X-Authority-Id", "X-Authority-UUID")):
        merged["headers"] = headers
        return merged

    payload = merged.get("json") if isinstance(merged.get("json"), dict) else {}
    authority_id = payload.get("authority_id") or AUTHORITY_ID
    if authority_id:
        headers[DEFAULT_AUTHORITY_HEADER] = authority_id
    merged["headers"] = headers
    return merged


def call_api(name: str, method: str, path: str, expected, base: str = MINTER_API_V1, timeout: int = 30, **kwargs):
    expected_codes = expected if isinstance(expected, (list, tuple, set)) else [expected]
    url = f"{base}{path}"
    request_kwargs = _with_authority_headers(method, base, kwargs)
    try:
        resp = requests.request(method=method.upper(), url=url, timeout=timeout, **request_kwargs)
    except Exception as exc:
        record(name, "FAIL", f"Request exception against {url}: {exc}")
        return None, None

    body = decode_body(resp)
    ok = resp.status_code in expected_codes
    detail = f"expected={list(expected_codes)} got={resp.status_code} method={method.upper()} url={url}"
    record(name, "PASS" if ok else "FAIL", detail)
    print(f"Status: {resp.status_code}")
    if isinstance(body, dict):
        pprint(body)
    else:
        text = str(body)
        print(text[:1000] + ("..." if len(text) > 1000 else ""))
    return resp, body



def describe_worker_status(status_body: dict):
    if not isinstance(status_body, dict):
        print("Worker status unavailable")
        return
    print("Worker overall:", status_body.get("overall"), "-", status_body.get("message"))
    for name, worker in (status_body.get("workers") or {}).items():
        cycle = worker.get("last_cycle") or {}
        queue = worker.get("queue") or {}
        success_label = "published" if name == "chain" else "persisted"
        print(
            f"{name}: state={worker.get('state')} alive={worker.get('alive')} "
            f"queue_pending={queue.get('pending')} queue_ready={queue.get('ready')} "
            f"last_cycle_attempted={cycle.get('processed')} "
            f"last_cycle_{success_label}={cycle.get('succeeded')} "
            f"failed={cycle.get('failed')} duration={cycle.get('duration_seconds')}s"
        )

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def minimal_metadata(title: str, version: int = 1):
    return {
        "title": title,
        "authors": ["Notebook Test"],
        "year": 2026,
        "version": version,
        "alternate_identifiers": [{"schema": "local", "value": f"{title}-{version}"}],
    }


def original_metadata_json(title: str, version: int = 1):
    return json.dumps({
        "title": title,
        "version": version,
        "generated_at": now_iso(),
        "source": "unified-notebook",
    }, ensure_ascii=True)


def original_metadata_xml(title: str, version: int = 1):
    return f"""<?xml version="1.0" encoding="UTF-8"?>
<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/"
           xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:title>{title}</dc:title>
  <dc:creator>Unified Notebook</dc:creator>
  <dc:description>Version {version}</dc:description>
  <dc:date>{now_iso()}</dc:date>
</oai_dc:dc>"""


def update_payload(title: str, target: str, version: int = 1, metadata_schema: str = "dublin_core", original_format: str = "json"):
    original_metadata = original_metadata_json(title, version)
    if original_format == "xml":
        original_metadata = original_metadata_xml(title, version)
    return {
        "authority_id": AUTHORITY_ID,
        "target": target,
        "minimal_metadata": minimal_metadata(title, version),
        "original_metadata": original_metadata,
        "metadata_schema": metadata_schema,
    }


def wait_for_authorized(timeout_seconds: int = POLL_TIMEOUT_SECONDS):
    started = time.time()
    last = None
    while time.time() - started <= timeout_seconds:
        try:
            resp = requests.get(f"{MINTER_API_V1}/authority/{AUTHORITY_ID}/authorized/{NAAN}", timeout=30)
            body = decode_body(resp)
            last = (resp.status_code, body)
            if resp.status_code == 200 and isinstance(body, dict) and body.get("authorized") is True:
                return True, last
        except Exception as exc:
            last = (None, str(exc))
        time.sleep(POLL_INTERVAL_SECONDS)
    return False, last


def wait_for_state(ark: str, expected_state: str, timeout_seconds: int = POLL_TIMEOUT_SECONDS):
    started = time.time()
    last = None
    while time.time() - started <= timeout_seconds:
        resp, body = call_api(
            name=f"Poll {ark} for state {expected_state}",
            method="GET",
            path=f"/arks/{ark}",
            expected=[200, 404],
        )
        last = body
        if resp is not None and resp.status_code == 200 and isinstance(body, dict) and body.get("state") == expected_state:
            return True, last
        time.sleep(POLL_INTERVAL_SECONDS)
    return False, last


## 1. Smoke Checks and Authority Bootstrap


In [ ]:
health_resp, health_body = call_api(
    name="Minter health endpoint reachable",
    method="GET",
    path="/health",
    expected=[200, 503],
    base=MINTER_BASE_URL,
)
check(
    name="Minter health payload has status",
    condition=isinstance(health_body, dict) and "status" in health_body,
    ok_detail=str(health_body),
    fail_detail=str(health_body),
)

worker_resp, worker_body = call_api(
    name="Worker status endpoint reachable",
    method="GET",
    path="/worker/status",
    expected=200,
)
workers = worker_body.get("workers", {}) if isinstance(worker_body, dict) else {}
describe_worker_status(worker_body)
metadata_worker_alive = bool((workers.get("metadata") or {}).get("alive"))
chain_worker_alive = bool((workers.get("chain") or {}).get("alive"))
worker_running = metadata_worker_alive and chain_worker_alive
CONTEXT["worker_running"] = worker_running
check(
    name="Worker status has expected shape",
    condition=isinstance(worker_body, dict) and "overall" in worker_body and "workers" in worker_body,
    ok_detail=f"metadata_alive={metadata_worker_alive} chain_alive={chain_worker_alive}",
    fail_detail=str(worker_body),
)

if REGISTER_AUTHORITY:
    call_api(
        name="Admin API status reachable",
        method="GET",
        path="/status",
        expected=200,
        base=ADMIN_API_URL,
    )

    payload = {"uuid": AUTHORITY_ID, "naans": [NAAN], "fund_amount_eth": 0.05}
    register_ok = False
    register_detail = ""
    for attempt in range(1, 3):
        try:
            resp = requests.post(f"{ADMIN_API_URL}/authority", json=payload, timeout=180)
            body = decode_body(resp)
            if resp.status_code in {200, 201, 409}:
                register_ok = True
                register_detail = f"status={resp.status_code} attempt={attempt}"
                break
            register_detail = f"status={resp.status_code} body={body} attempt={attempt}"
        except Exception as exc:
            register_detail = f"attempt={attempt} exception={exc}"
        time.sleep(POLL_INTERVAL_SECONDS)
    record("Admin register authority", "PASS" if register_ok else "FAIL", register_detail)

    authorized_now, last_auth = wait_for_authorized(timeout_seconds=30)
    if not authorized_now:
        auth_resp = requests.post(
            f"{ADMIN_API_URL}/authority/{AUTHORITY_ID}/authorize-naan",
            json={"naan": NAAN},
            timeout=120,
        )
        auth_body = decode_body(auth_resp)
        record(
            "Admin authorize NAAN fallback",
            "PASS" if auth_resp.status_code == 200 else "FAIL",
            f"status={auth_resp.status_code} body={auth_body}",
        )
else:
    skip("Admin API status reachable", "REGISTER_AUTHORITY=false")
    skip("Admin register authority", "REGISTER_AUTHORITY=false")

call_api("Authority details", "GET", f"/authority/{AUTHORITY_ID}", expected=200)
_, naans_body = call_api("Authority NAAN list", "GET", f"/authority/{AUTHORITY_ID}/naans", expected=200)
authorized_ok, last_auth = wait_for_authorized()
record("Authority authorization check", "PASS" if authorized_ok else "FAIL", f"last_auth={last_auth}")
check(
    name="Authority NAAN list contains configured NAAN",
    condition=isinstance(naans_body, dict) and NAAN in (naans_body.get("naans") or []),
    ok_detail=str(naans_body),
    fail_detail=str(naans_body),
)


## 2. Reserve and Representative Validation


In [ ]:
_, reserve_body = call_api(
    name="Reserve single ARK",
    method="POST",
    path="/arks",
    expected=201,
    json={"authority_id": AUTHORITY_ID, "naan": NAAN},
)
ark_reserved = reserve_body.get("ark") if isinstance(reserve_body, dict) else None
CONTEXT["ark_reserved"] = ark_reserved
check(
    name="Reserve returned ARK identifier",
    condition=bool(ark_reserved),
    ok_detail=str(ark_reserved),
    fail_detail=str(reserve_body),
)
if isinstance(reserve_body, dict):
    check(
        name="Reserved ARK state is RESERVED",
        condition=reserve_body.get("state") == "R",
        ok_detail=str(reserve_body),
        fail_detail=str(reserve_body),
    )

if ark_reserved:
    _, reserved_get_body = call_api(
        name="Get reserved ARK",
        method="GET",
        path=f"/arks/{ark_reserved}",
        expected=200,
    )
    check(
        name="Get reserved ARK returns same identifier",
        condition=isinstance(reserved_get_body, dict) and reserved_get_body.get("ark") == ark_reserved,
        ok_detail=str(reserved_get_body),
        fail_detail=str(reserved_get_body),
    )

call_api(
    name="Reserve rejects deprecated top-level alternate_identifiers",
    method="POST",
    path="/arks",
    expected=422,
    json={
        "authority_id": AUTHORITY_ID,
        "naan": NAAN,
        "alternate_identifiers": [{"schema": "doi", "value": "10.1234/test"}],
    },
)

if UNAUTHORIZED_NAAN == NAAN:
    skip("Unauthorized reserve is rejected", "UNAUTHORIZED_NAAN matches NAAN; set a different env value to run this check")
else:
    call_api(
        name="Unauthorized reserve is rejected",
        method="POST",
        path="/arks",
        expected=403,
        json={"authority_id": AUTHORITY_ID, "naan": UNAUTHORIZED_NAAN},
    )


## 3. Update to DRAFT with JSON and XML Metadata


In [ ]:
ark = CONTEXT.get("ark_reserved")
if not ark:
    skip("JSON update to DRAFT", "No reserved ARK available")
else:
    _, draft_body = call_api(
        name="Update reserved ARK to DRAFT (JSON)",
        method="PUT",
        path=f"/arks/{ark}",
        expected=200,
        json=update_payload(
            title="Unified notebook JSON draft",
            version=1,
            target="https://example.org/unified/json-v1",
        ),
    )
    if isinstance(draft_body, dict):
        check("ARK state is DRAFT after JSON update", draft_body.get("state") == "D", str(draft_body), str(draft_body))
        check("JSON update preserved metadata schema", draft_body.get("metadata_schema") == "dublin_core", str(draft_body), str(draft_body))
        title = ((draft_body.get("minimal_metadata") or {}).get("title"))
        check("JSON update stored minimal metadata title", title == "Unified notebook JSON draft", str(draft_body), str(draft_body))

    _, draft_overwrite_body = call_api(
        name="Reject pending DRAFT overwrite",
        method="PUT",
        path=f"/arks/{ark}",
        expected=409,
        json=update_payload(
            title="Unified notebook JSON overwrite",
            version=2,
            target="https://example.org/unified/json-v2",
        ),
    )
    if isinstance(draft_overwrite_body, dict):
        detail = str(draft_overwrite_body.get("detail", "")).lower()
        check("Pending DRAFT overwrite is rejected", "pending creation" in detail, str(draft_overwrite_body), str(draft_overwrite_body))

    call_api(
        name="Update rejects deprecated top-level alternate_identifiers",
        method="PUT",
        path=f"/arks/{ark}",
        expected=422,
        json={
            **update_payload(
                title="Bad alternate identifiers",
                version=3,
                target="https://example.org/unified/json-bad",
            ),
            "alternate_identifiers": [{"schema": "doi", "value": "10.1234/bad"}],
        },
    )

_, xml_reserve_body = call_api(
    name="Reserve ARK for XML update",
    method="POST",
    path="/arks",
    expected=201,
    json={"authority_id": AUTHORITY_ID, "naan": NAAN},
)
ark_xml = xml_reserve_body.get("ark") if isinstance(xml_reserve_body, dict) else None
if not ark_xml:
    skip("XML update to DRAFT", "No XML ARK reserved")
else:
    _, xml_body = call_api(
        name="Update reserved ARK to DRAFT (XML)",
        method="PUT",
        path=f"/arks/{ark_xml}",
        expected=200,
        json=update_payload(
            title="Unified notebook XML draft",
            version=1,
            target="https://example.org/unified/xml-v1",
            metadata_schema="oai_dc",
            original_format="xml",
        ),
    )
    if isinstance(xml_body, dict):
        check("XML update reaches DRAFT", xml_body.get("state") == "D", str(xml_body), str(xml_body))
        check("XML update keeps oai_dc schema", xml_body.get("metadata_schema") == "oai_dc", str(xml_body), str(xml_body))


## 4. Batch Reserve and Tombstone Lifecycle


In [ ]:
_, batch_body = call_api(
    name="Batch reserve returns results",
    method="POST",
    path="/arks/batch",
    expected=200,
    json={
        "authority_id": AUTHORITY_ID,
        "naan": NAAN,
        "items": [
            {"client_item_id": "req-001"},
            {"client_item_id": "req-002"},
        ],
    },
)
batch_results = batch_body.get("results") if isinstance(batch_body, dict) else None
check("Batch reserve returned two results", isinstance(batch_results, list) and len(batch_results) == 2, str(batch_body), str(batch_body))
if isinstance(batch_results, list):
    arks = [row.get("ark") for row in batch_results if isinstance(row, dict)]
    check("Batch reserve generated unique ARKs", len(arks) == len(set(arks)) == 2, str(batch_results), str(batch_results))

ark_tombstone = CONTEXT.get("ark_reserved")
if not ark_tombstone:
    skip("Tombstone flow", "No reserved ARK available")
else:
    call_api("Delete ARK to TOMBSTONE", "DELETE", f"/arks/{ark_tombstone}", expected=[200, 204])
    _, tombstone_body = call_api("Get tombstoned ARK", "GET", f"/arks/{ark_tombstone}", expected=200)
    if isinstance(tombstone_body, dict):
        check("ARK state is TOMBSTONE", tombstone_body.get("state") == "T", str(tombstone_body), str(tombstone_body))
    call_api("Delete tombstoned ARK remains idempotent", "DELETE", f"/arks/{ark_tombstone}", expected=[200, 204])
    call_api(
        name="Reject update on tombstoned ARK",
        method="PUT",
        path=f"/arks/{ark_tombstone}",
        expected=409,
        json=update_payload(
            title="blocked tombstone update",
            version=99,
            target="https://example.org/blocked",
        ),
    )


## 5. Optional Worker Publish/Update and Optional Chain Import

These sections stay in the unified notebook because they reflect current behavior, but they are optional in practice:
- worker flow only runs when `/api/v1/worker/status` reports running metadata and chain workers
- chain import only runs when `EXISTING_CHAIN_ARK` is provided


In [ ]:
if not CONTEXT.get("worker_running"):
    skip("Worker publish/update flow", "Metadata and chain workers are not both running according to /worker/status")
else:
    _, worker_reserve_body = call_api(
        name="Worker flow reserve",
        method="POST",
        path="/arks",
        expected=201,
        json={"authority_id": AUTHORITY_ID, "naan": NAAN},
    )
    ark_worker = worker_reserve_body.get("ark") if isinstance(worker_reserve_body, dict) else None
    if not ark_worker:
        record("Worker flow reserve returned ARK", "FAIL", str(worker_reserve_body))
    else:
        call_api(
            name="Worker flow set DRAFT",
            method="PUT",
            path=f"/arks/{ark_worker}",
            expected=200,
            json=update_payload(
                title="Worker create flow",
                version=1,
                target="https://example.org/worker-create",
            ),
        )
        ok_published, published_body = wait_for_state(ark_worker, expected_state="P")
        check("Workers transition DRAFT to PUBLISHED", ok_published, str(published_body), f"Timed out waiting for PUBLISHED. Last body={published_body}")

        _, update_body = call_api(
            name="Published ARK transitions to UPDATE",
            method="PUT",
            path=f"/arks/{ark_worker}",
            expected=200,
            json=update_payload(
                title="Worker update flow",
                version=2,
                target="https://example.org/worker-update",
            ),
        )
        if isinstance(update_body, dict):
            check("Published ARK moved to UPDATE", update_body.get("state") == "U", str(update_body), str(update_body))

        ok_republished, republished_body = wait_for_state(ark_worker, expected_state="P")
        check("Workers transition UPDATE back to PUBLISHED", ok_republished, str(republished_body), f"Timed out waiting for republish. Last body={republished_body}")

if not EXISTING_CHAIN_ARK:
    skip("Chain import flow", "EXISTING_CHAIN_ARK is not set")
else:
    _, import_body = call_api(
        name="PUT imports on-chain ARK then transitions to UPDATE",
        method="PUT",
        path=f"/arks/{EXISTING_CHAIN_ARK}",
        expected=200,
        json=update_payload(
            title="Imported then updated",
            version=1,
            target="https://example.org/imported-update",
        ),
    )
    if isinstance(import_body, dict):
        check("Imported ARK ends in UPDATE", import_body.get("state") == "U", str(import_body), str(import_body))


## Summary

This notebook aims to be the clearest single entry point for Minter integration testing.
For deeper debugging or stress scenarios, keep using the focused notebooks.


In [ ]:
summary = summarize_results()

if summary["failed"] > 0:
    raise AssertionError(f"There are {summary['failed']} failed checks. Review notebook output.")
else:
    print("All checked scenarios passed.")
